In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# --------------------------------------------------
# Find project root
# --------------------------------------------------

cwd = Path.cwd()

if (cwd / "tables1").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "tables1").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root.")

print("PROJECT_ROOT:", PROJECT_ROOT)

# --------------------------------------------------
# Load YFF Bayesian Lambda matrix
# --------------------------------------------------

lambda_dir = (
    PROJECT_ROOT
    / "tables1"
    / "gamma_poisson"
    / "lambda_matrices"
)

lambda_path = lambda_dir / "YFF_lambda.npy"

Lambda = np.load(lambda_path)

print("\nYFF Lambda loaded successfully.")
print("Shape:", Lambda.shape)
print("Data type:", Lambda.dtype)

# --------------------------------------------------
# Basic checks
# --------------------------------------------------

print("\nBasic checks")
print("All finite:", np.isfinite(Lambda).all())
print("Any negative values:", (Lambda < 0).any())

print("\nValue summary")
print("Minimum:", Lambda.min())
print("Maximum:", Lambda.max())
print("Mean:", Lambda.mean())
print("Median:", np.median(Lambda))

# --------------------------------------------------
# Mean posterior rate for each number
# --------------------------------------------------

row_means = Lambda.mean(axis=1)

print("\nMean Lambda across features for each number")

for number, value in enumerate(row_means, start=1):
    print(f"Number {number}: {value:.6f}")

PROJECT_ROOT: c:\Users\shafi\number-simplex-reproduction

YFF Lambda loaded successfully.
Shape: (9, 333)
Data type: float64

Basic checks
All finite: True
Any negative values: False

Value summary
Minimum: 0.011111111111111112
Maximum: 2.7038461538461536
Mean: 0.6487734143954188
Median: 0.7038461538461539

Mean Lambda across features for each number
Number 1: 0.598634
Number 2: 0.799748
Number 3: 0.632904
Number 4: 0.550062
Number 5: 0.474457
Number 6: 0.714472
Number 7: 0.730497
Number 8: 0.665152
Number 9: 0.673036


In [2]:
# --------------------------------------------------
# Step 2: Center the Bayesian number representations
# --------------------------------------------------

# Centroid of the 9 number representations
lambda_centroid = Lambda.mean(axis=0, keepdims=True)

# Center each number representation
C = Lambda - lambda_centroid

print("Original Lambda shape:", Lambda.shape)
print("Centered matrix C shape:", C.shape)

# --------------------------------------------------
# Verify centering
# --------------------------------------------------

print("\nMaximum absolute column mean after centering:")
print(np.abs(C.mean(axis=0)).max())

# --------------------------------------------------
# Distance of each number from the global centroid
# --------------------------------------------------

centroid_distances = np.linalg.norm(C, axis=1)

print("\nDistance of each number from the centroid:")

for number, distance in enumerate(centroid_distances, start=1):
    print(f"Number {number}: {distance:.6f}")

print("\nCentroid-distance summary")
print("Mean:", centroid_distances.mean())
print("SD:", centroid_distances.std())
print("Minimum:", centroid_distances.min())
print("Maximum:", centroid_distances.max())

Original Lambda shape: (9, 333)
Centered matrix C shape: (9, 333)

Maximum absolute column mean after centering:
2.837236618486511e-16

Distance of each number from the centroid:
Number 1: 3.275169
Number 2: 4.695488
Number 3: 3.030523
Number 4: 4.618215
Number 5: 4.850224
Number 6: 3.587604
Number 7: 3.785579
Number 8: 2.970600
Number 9: 3.192666

Centroid-distance summary
Mean: 3.7784520895877276
SD: 0.7102257848015948
Minimum: 2.9706002616630567
Maximum: 4.850223926730397


In [3]:
# --------------------------------------------------
# Step 3: SVD and dimensionality of Bayesian geometry
# --------------------------------------------------

U, singular_values, Vt = np.linalg.svd(
    C,
    full_matrices=False
)

# Variance associated with each singular direction
variance = singular_values**2

explained_variance_ratio = variance / variance.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)

# Numerical matrix rank
numerical_rank = np.linalg.matrix_rank(C)

print("Centered matrix shape:", C.shape)
print("Maximum possible affine rank:", 8)
print("Numerical rank:", numerical_rank)

print("\nSingular-value spectrum")

for i, (s, evr, cum) in enumerate(
    zip(
        singular_values,
        explained_variance_ratio,
        cumulative_variance
    ),
    start=1
):
    print(
        f"Dimension {i}: "
        f"singular value = {s:.6f}, "
        f"variance = {100 * evr:.2f}%, "
        f"cumulative = {100 * cum:.2f}%"
    )

# --------------------------------------------------
# Effective dimensionality
# --------------------------------------------------

p = explained_variance_ratio[
    explained_variance_ratio > 0
]

effective_rank = np.exp(
    -np.sum(p * np.log(p))
)

participation_ratio = 1 / np.sum(p**2)

print("\nEffective dimensionality")
print("Effective rank:", effective_rank)
print("Participation ratio:", participation_ratio)

Centered matrix shape: (9, 333)
Maximum possible affine rank: 8
Numerical rank: 8

Singular-value spectrum
Dimension 1: singular value = 7.185431, variance = 38.81%, cumulative = 38.81%
Dimension 2: singular value = 4.408285, variance = 14.61%, cumulative = 53.42%
Dimension 3: singular value = 3.679592, variance = 10.18%, cumulative = 63.60%
Dimension 4: singular value = 3.431431, variance = 8.85%, cumulative = 72.45%
Dimension 5: singular value = 3.292445, variance = 8.15%, cumulative = 80.60%
Dimension 6: singular value = 3.138308, variance = 7.40%, cumulative = 88.00%
Dimension 7: singular value = 2.915446, variance = 6.39%, cumulative = 94.39%
Dimension 8: singular value = 2.731954, variance = 5.61%, cumulative = 100.00%
Dimension 9: singular value = 0.000000, variance = 0.00%, cumulative = 100.00%

Effective dimensionality
Effective rank: 6.233224623969974
Participation ratio: 4.772958494323755


In [4]:
# --------------------------------------------------
# Step 3: SVD and dimensionality
# --------------------------------------------------

U, singular_values, Vt = np.linalg.svd(
    C,
    full_matrices=False
)

# Variance carried by each singular direction
variance = singular_values**2

explained_variance_ratio = variance / variance.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)

# Numerical rank
numerical_rank = np.linalg.matrix_rank(C)

print("Centered matrix shape:", C.shape)
print("Maximum possible affine rank:", 8)
print("Numerical rank:", numerical_rank)

print("\nSingular-value spectrum")

for i, (s, evr, cum) in enumerate(
    zip(
        singular_values,
        explained_variance_ratio,
        cumulative_variance
    ),
    start=1
):
    print(
        f"Dimension {i}: "
        f"singular value = {s:.6f}, "
        f"variance = {100 * evr:.2f}%, "
        f"cumulative = {100 * cum:.2f}%"
    )

# --------------------------------------------------
# Effective dimensionality
# --------------------------------------------------

# Ignore the zero singular direction created by centering
p = explained_variance_ratio[
    explained_variance_ratio > 1e-12
]

effective_rank = np.exp(
    -np.sum(p * np.log(p))
)

participation_ratio = 1 / np.sum(p**2)

print("\nEffective dimensionality")
print("Effective rank:", effective_rank)
print("Participation ratio:", participation_ratio)

Centered matrix shape: (9, 333)
Maximum possible affine rank: 8
Numerical rank: 8

Singular-value spectrum
Dimension 1: singular value = 7.185431, variance = 38.81%, cumulative = 38.81%
Dimension 2: singular value = 4.408285, variance = 14.61%, cumulative = 53.42%
Dimension 3: singular value = 3.679592, variance = 10.18%, cumulative = 63.60%
Dimension 4: singular value = 3.431431, variance = 8.85%, cumulative = 72.45%
Dimension 5: singular value = 3.292445, variance = 8.15%, cumulative = 80.60%
Dimension 6: singular value = 3.138308, variance = 7.40%, cumulative = 88.00%
Dimension 7: singular value = 2.915446, variance = 6.39%, cumulative = 94.39%
Dimension 8: singular value = 2.731954, variance = 5.61%, cumulative = 100.00%
Dimension 9: singular value = 0.000000, variance = 0.00%, cumulative = 100.00%

Effective dimensionality
Effective rank: 6.233224623969974
Participation ratio: 4.772958494323755


In [5]:
# --------------------------------------------------
# Step 4: Pairwise Euclidean distances
# --------------------------------------------------

# Compute pairwise differences between all 9 number vectors
differences = Lambda[:, None, :] - Lambda[None, :, :]

# Euclidean norm across the 333 features
D = np.linalg.norm(differences, axis=2)

# Labels 1 through 9
numbers = np.arange(1, 10)

# Put distance matrix into a readable table
D_df = pd.DataFrame(
    D,
    index=numbers,
    columns=numbers
)

print("Pairwise Euclidean distance matrix:\n")
print(D_df.round(3))

# --------------------------------------------------
# Extract the 36 unique distances
# --------------------------------------------------

upper_triangle = np.triu_indices(9, k=1)

edge_distances = D[upper_triangle]

print("\nPairwise-distance summary")
print("Number of unique pairs:", len(edge_distances))
print("Mean distance:", edge_distances.mean())
print("SD:", edge_distances.std())
print("Minimum:", edge_distances.min())
print("Maximum:", edge_distances.max())

# Relative variability of pairwise distances
edge_cv = edge_distances.std() / edge_distances.mean()

print("Coefficient of variation:", edge_cv)
print(
    "Maximum / Minimum:",
    edge_distances.max() / edge_distances.min()
)

# --------------------------------------------------
# Identify closest and farthest pairs
# --------------------------------------------------

pair_i = upper_triangle[0]
pair_j = upper_triangle[1]

min_idx = np.argmin(edge_distances)
max_idx = np.argmax(edge_distances)

closest_pair = (
    pair_i[min_idx] + 1,
    pair_j[min_idx] + 1
)

farthest_pair = (
    pair_i[max_idx] + 1,
    pair_j[max_idx] + 1
)

print("\nClosest pair:", closest_pair)
print(
    "Closest distance:",
    edge_distances[min_idx]
)

print("\nFarthest pair:", farthest_pair)
print(
    "Farthest distance:",
    edge_distances[max_idx]
)

Pairwise Euclidean distance matrix:

       1      2      3      4      5      6      7      8      9
1  0.000  6.565  4.656  5.300  5.377  5.408  5.616  4.679  5.012
2  6.565  0.000  6.282  7.735  8.815  5.335  4.961  5.456  5.342
3  4.656  6.282  0.000  5.650  5.404  4.784  5.175  4.476  4.870
4  5.300  7.735  5.650  0.000  5.741  6.825  7.253  5.835  6.242
5  5.377  8.815  5.404  5.741  0.000  7.060  7.089  6.074  6.244
6  5.408  5.335  4.784  6.825  7.060  0.000  4.999  4.756  4.924
7  5.616  4.961  5.175  7.253  7.089  4.999  0.000  5.159  4.961
8  4.679  5.456  4.476  5.835  6.074  4.756  5.159  0.000  4.537
9  5.012  5.342  4.870  6.242  6.244  4.924  4.961  4.537  0.000

Pairwise-distance summary
Number of unique pairs: 36
Mean distance: 5.683182231958062
SD: 0.9792658732166267
Minimum: 4.476097515718541
Maximum: 8.814638745941506
Coefficient of variation: 0.17230942687530787
Maximum / Minimum: 1.9692687022540227

Closest pair: (np.int64(3), np.int64(8))
Closest distance: 4.476

In [6]:
np.linalg.matrix_rank(D)

np.int64(9)